# Delay Timing Verification

Stage 4 delay sanity plots: resolved sync divisions at 120 BPM, impulse repeat positions, and filtered feedback decay. These mirror `Source/DSP/DelayConfig.h` and `Source/DSP/Delay.cpp`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample_rate = 48_000
tempo_bpm = 120.0
feedback = 0.5
feedback_lowpass_hz = 4_000.0

divisions = {
    "1/4": 1.0,
    "1/4 dotted": 1.5,
    "1/4 triplet": 2.0 / 3.0,
    "1/8": 0.5,
    "1/8 dotted": 0.75,
    "1/8 triplet": 1.0 / 3.0,
    "1/16": 0.25,
    "1/16 dotted": 0.375,
    "1/16 triplet": 1.0 / 6.0,
}

def delay_samples(beats, bpm=tempo_bpm):
    return int(round((60.0 / bpm) * beats * sample_rate))

labels = list(divisions.keys())
sample_counts = [delay_samples(beats) for beats in divisions.values()]

plt.figure(figsize=(10, 4))
plt.bar(labels, np.array(sample_counts) / sample_rate * 1000.0)
plt.ylabel("Delay time (ms)")
plt.title("Resolved Stage 4 sync divisions at 120 BPM")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
delay = delay_samples(divisions["1/8 dotted"])
length = delay * 4
signal = np.zeros(length)
signal[0] = 1.0
feedback_state = 0.0
lowpass_coefficient = 1.0 - np.exp(-2.0 * np.pi * feedback_lowpass_hz / sample_rate)
delay_line = np.zeros(length)
output = np.zeros(length)

for index in range(length):
    delayed = delay_line[index - delay] if index >= delay else 0.0
    feedback_state += lowpass_coefficient * (delayed - feedback_state)
    delay_line[index] = signal[index] + feedback * feedback_state
    output[index] = delayed

time_ms = np.arange(length) / sample_rate * 1000.0
plt.figure(figsize=(10, 4))
plt.plot(time_ms, output)
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude")
plt.title("Dotted 1/8 impulse repeats with filtered feedback")
plt.xlim(0, time_ms[-1])
plt.tight_layout()
plt.show()
